# Interactive LOOP vs DIFFABLE_TIME_DOMAIN Comparison

Compare the two implementations of the physical model with LFO modulation.

**Features:**

- LFO with controllable frequency and amplitude
- Independent parameter control for all synthesis parameters
- Route LFO to any combination of parameters
- Visual comparison: spectrograms, waveforms, parameter trajectories
- Numerical comparison: SNR, RMS error, max absolute difference

In [1]:
import numpy as np
import numpy.typing as npt
from dataclasses import dataclass, field
from typing import Optional
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from scipy import signal
import librosa
import IPython.display as ipd

from synths import (
    physical_model, 
    Implementation,
    F0_MIN,
    FS_MIN, 
    RND_SEED, 
    LAGRANGE_ORDER,
    IIR_TRUNCATION,
    ONSET_THRESHOLD
)

/Users/pablotablasdepaula/PycharmProjects/DAFx26-Karplus/.pixi/envs/default/lib/python3.13/site-packages/philtorch/__init__.py:11: UserWarning: Custom extension not loaded.
  warnings.warn("Custom extension not loaded.")
/Users/pablotablasdepaula/PycharmProjects/DAFx26-Karplus/.pixi/envs/default/lib/python3.13/site-packages/torchlpc/__init__.py:23: UserWarning: Custom extension not loaded. Falling back to Numba implementation.
  warnings.warn("Custom extension not loaded. Falling back to Numba implementation.")


In [2]:
@dataclass
class ParameterRange:
    """Defines valid range for a parameter."""
    min: float
    max: float

    def clip(self, value: npt.NDArray) -> npt.NDArray:
        return np.clip(value, self.min, self.max)

# Single source of truth for parameter ranges
PARAM_RANGES = {
    'f0': ParameterRange(F0_MIN, FS_MIN / 2.0),
    'pluck_position': ParameterRange(0.0, 1.0),
    'burst_gain': ParameterRange(0.0, 1.0),
    'dynamic_level': ParameterRange(0.0, 1.0),
    'a1': ParameterRange(0.0, 1.0),
    'decay': ParameterRange(0.0, 1.0),
}

@dataclass
class LFOConfig:
    """Low Frequency Oscillator configuration."""
    rate: float  # Hz
    amplitude: float  # 0-1, fraction of parameter range

    def generate(self, num_frames: int, center: float, param_range: ParameterRange, duration: float) -> npt.NDArray:
        """Generate LFO-modulated parameter trajectory."""
        t = np.linspace(0, duration, num_frames)
        lfo = np.sin(2 * np.pi * self.rate * t)
        range_span = param_range.max - param_range.min
        modulation_depth = self.amplitude * range_span / 2.0
        return param_range.clip(center + lfo * modulation_depth)

@dataclass
class ExperimentConfig:
    """Configuration for synthesis experiment."""
    duration: float
    trigger_rate: float
    num_frames: int = 250
    fs: int = FS_MIN
    
    # Parameter centers
    f0_center: float = 440.0
    pluck_position_center: float = 0.5
    burst_gain_center: float = 0.5
    dynamic_level_center: float = 0.5
    a1_center: float = 0.5
    decay_center: float = 0.99
    
    # LFO assignments (None = no modulation)
    f0_lfo: Optional[LFOConfig] = None
    pluck_position_lfo: Optional[LFOConfig] = None
    burst_gain_lfo: Optional[LFOConfig] = None
    dynamic_level_lfo: Optional[LFOConfig] = None
    a1_lfo: Optional[LFOConfig] = None
    decay_lfo: Optional[LFOConfig] = None

    def get_signal_length(self) -> int:
        return int(self.duration * self.fs)

    def generate_onset_probs(self) -> torch.Tensor:
        """Generate onset probability tensor with triggers at regular intervals."""
        num_triggers = int(self.duration * self.trigger_rate)
        trigger_interval = self.num_frames / (self.duration * self.trigger_rate)
        
        onset_probs = np.zeros(self.num_frames)
        for i in range(num_triggers):
            frame_idx = int(i * trigger_interval)
            if frame_idx < self.num_frames:
                onset_probs[frame_idx] = 1.0
        
        return torch.from_numpy(onset_probs).float().unsqueeze(0)

    def generate_parameter(self, param_name: str, center: float, lfo: Optional[LFOConfig]) -> npt.NDArray:
        """Generate parameter trajectory (constant or LFO-modulated)."""
        param_range = PARAM_RANGES[param_name]
        if lfo is None:
            return np.full(self.num_frames, param_range.clip(np.array([center]))[0])
        return lfo.generate(self.num_frames, center, param_range, self.duration)

    def get_all_params(self) -> dict[str, npt.NDArray]:
        """Generate all parameter trajectories."""
        return {
            'f0': self.generate_parameter('f0', self.f0_center, self.f0_lfo),
            'pluck_position': self.generate_parameter('pluck_position', self.pluck_position_center, self.pluck_position_lfo),
            'burst_gain': self.generate_parameter('burst_gain', self.burst_gain_center, self.burst_gain_lfo),
            'dynamic_level': self.generate_parameter('dynamic_level', self.dynamic_level_center, self.dynamic_level_lfo),
            'a1': self.generate_parameter('a1', self.a1_center, self.a1_lfo),
            'decay': self.generate_parameter('decay', self.decay_center, self.decay_lfo),
        }

    def synthesize(self, implementation: Implementation) -> tuple[npt.NDArray, dict]:
        """Synthesize audio using specified implementation."""
        params_np = self.get_all_params()
        onset_probs = self.generate_onset_probs()
        
        # Convert to torch tensors
        params_torch = {
            k: torch.from_numpy(v).float().unsqueeze(0) 
            for k, v in params_np.items()
        }
        
        audio = physical_model(
            onset_probs=onset_probs,
            num_samples=self.get_signal_length(),
            implementation=implementation,
            fs=self.fs,
            training=False,
            **params_torch
        )
        
        return audio.squeeze(0).numpy(), params_np

In [3]:
# Layout for sliders
slider_layout = widgets.Layout(width='400px')
checkbox_layout = widgets.Layout(width='100px', margin='20px 0 0 10px')

# LFO Controls
lfo_rate = widgets.FloatSlider(
    value=1.0, min=0.1, max=20.0, step=0.1,
    description='LFO Rate (Hz):', layout=slider_layout,
    style={'description_width': '120px'}
)
lfo_amplitude = widgets.FloatSlider(
    value=0.3, min=0.0, max=1.0, step=0.01,
    description='LFO Amplitude:', layout=slider_layout,
    style={'description_width': '120px'}
)

# General Controls
trigger_rate = widgets.FloatSlider(
    value=4.0, min=0.5, max=20.0, step=0.5,
    description='Trigger Rate (Hz):', layout=slider_layout,
    style={'description_width': '120px'}
)
duration_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=8.0, step=0.5,
    description='Duration (s):', layout=slider_layout,
    style={'description_width': '120px'}
)

# Parameter Centers and LFO Routing (using PARAM_RANGES for consistency)
f0_center = widgets.FloatSlider(
    value=220.0, 
    min=PARAM_RANGES['f0'].min, 
    max=min(880.0, PARAM_RANGES['f0'].max), 
    step=1.0,
    description='F0 (Hz):', 
    layout=slider_layout,
    style={'description_width': '120px'}
)
f0_modulated = widgets.Checkbox(
    value=False, 
    description='🌊 Route LFO',
    layout=checkbox_layout,
    style={'description_width': 'initial'}
)

pluck_position_center = widgets.FloatSlider(
    value=0.5, 
    min=PARAM_RANGES['pluck_position'].min, 
    max=PARAM_RANGES['pluck_position'].max, 
    step=0.01,
    description='Pluck Position:', 
    layout=slider_layout,
    style={'description_width': '120px'}
)
pluck_position_modulated = widgets.Checkbox(
    value=False, 
    description='🌊 Route LFO',
    layout=checkbox_layout,
    style={'description_width': 'initial'}
)

burst_gain_center = widgets.FloatSlider(
    value=0.5, 
    min=PARAM_RANGES['burst_gain'].min, 
    max=PARAM_RANGES['burst_gain'].max, 
    step=0.01,
    description='Burst Gain:', 
    layout=slider_layout,
    style={'description_width': '120px'}
)
burst_gain_modulated = widgets.Checkbox(
    value=False, 
    description='🌊 Route LFO',
    layout=checkbox_layout,
    style={'description_width': 'initial'}
)

dynamic_level_center = widgets.FloatSlider(
    value=0.5, 
    min=PARAM_RANGES['dynamic_level'].min, 
    max=PARAM_RANGES['dynamic_level'].max, 
    step=0.01,
    description='Dynamic Level:', 
    layout=slider_layout,
    style={'description_width': '120px'}
)
dynamic_level_modulated = widgets.Checkbox(
    value=False, 
    description='🌊 Route LFO',
    layout=checkbox_layout,
    style={'description_width': 'initial'}
)

a1_center = widgets.FloatSlider(
    value=0.5, 
    min=PARAM_RANGES['a1'].min, 
    max=PARAM_RANGES['a1'].max, 
    step=0.01,
    description='Loop Filter (a1):', 
    layout=slider_layout,
    style={'description_width': '120px'}
)
a1_modulated = widgets.Checkbox(
    value=False, 
    description='🌊 Route LFO',
    layout=checkbox_layout,
    style={'description_width': 'initial'}
)

decay_center = widgets.FloatSlider(
    value=0.995, 
    min=PARAM_RANGES['decay'].min, 
    max=PARAM_RANGES['decay'].max, 
    step=0.0001,
    description='Decay:', 
    layout=slider_layout,
    style={'description_width': '120px'},
    readout_format='.4f'
)
decay_modulated = widgets.Checkbox(
    value=False, 
    description='🌊 Route LFO',
    layout=checkbox_layout,
    style={'description_width': 'initial'}
)

# Generate Button and Output
generate_button = widgets.Button(
    description='🎸 Generate & Compare',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)
output = widgets.Output()

In [4]:
def generate_comparison(b):
    with output:
        clear_output(wait=True)
        print("🎵 Generating audio with both implementations...")
        
        # Build LFO config
        lfo = LFOConfig(rate=lfo_rate.value, amplitude=lfo_amplitude.value)
        
        # Build experiment config
        config = ExperimentConfig(
            duration=duration_slider.value,
            trigger_rate=trigger_rate.value,
            num_frames=250,
            fs=FS_MIN,
            f0_center=f0_center.value,
            pluck_position_center=pluck_position_center.value,
            burst_gain_center=burst_gain_center.value,
            dynamic_level_center=dynamic_level_center.value,
            a1_center=a1_center.value,
            decay_center=decay_center.value,
            f0_lfo=lfo if f0_modulated.value else None,
            pluck_position_lfo=lfo if pluck_position_modulated.value else None,
            burst_gain_lfo=lfo if burst_gain_modulated.value else None,
            dynamic_level_lfo=lfo if dynamic_level_modulated.value else None,
            a1_lfo=lfo if a1_modulated.value else None,
            decay_lfo=lfo if decay_modulated.value else None,
        )
        
        # Synthesize with both implementations
        print("  → Running LOOP implementation...")
        audio_loop, params = config.synthesize(Implementation.LOOP)
        
        print("  → Running DIFFABLE_TIME_DOMAIN implementation...")
        audio_diff_td, _ = config.synthesize(Implementation.DIFFABLE_TIME_DOMAIN)
        
        # Time axes
        time_frames = np.linspace(0, config.duration, config.num_frames)
        time_samples = np.arange(config.get_signal_length()) / config.fs
        
        # Compute f0 using librosa pyin
        print("  → Computing f0 with librosa pyin...")
        f0_pyin, voiced_flag, voiced_probs = librosa.pyin(
            audio_loop, 
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=config.fs,
            frame_length=2048
        )
        # Resample to match parameter frame rate
        if len(f0_pyin) > 0:
            time_pyin = np.linspace(0, config.duration, len(f0_pyin))
            f0_pyin_resampled = np.interp(time_frames, time_pyin, np.nan_to_num(f0_pyin, nan=0.0))
        else:
            f0_pyin_resampled = np.zeros_like(time_frames)
        
        # Compute spectrograms
        print("  → Computing spectrograms...")
        nperseg = min(1024, config.get_signal_length() // 4)
        f_l, t_l, S_l = signal.spectrogram(audio_loop, config.fs, nperseg=nperseg)
        f_d, t_d, S_d = signal.spectrogram(audio_diff_td, config.fs, nperseg=nperseg)
        
        S_l_db = 10 * np.log10(S_l + 1e-10)
        S_d_db = 10 * np.log10(S_d + 1e-10)
        
        # ===== PLOTTING =====
        print("  → Generating plots...")
        fig = plt.figure(figsize=(14, 16))
        gs = fig.add_gridspec(6, 2, hspace=0.4, wspace=0.3)
        
        # Row 0: Waveform overlay (spanning both columns)
        ax_overlay = fig.add_subplot(gs[0, :])
        ax_overlay.plot(time_samples, audio_loop, 'b-', lw=1, label='LOOP', alpha=0.7)
        ax_overlay.plot(time_samples, audio_diff_td, 'orange', lw=1, ls='--', label='DIFFABLE_TD', alpha=0.7)
        ax_overlay.set_title('Waveform Overlay (Full)', fontweight='bold', fontsize=12)
        ax_overlay.set_xlabel('Time (s)', fontsize=10)
        ax_overlay.set_ylabel('Amplitude')
        ax_overlay.legend(loc='upper right', fontsize=9)
        ax_overlay.grid(True, alpha=0.3)
        
        # Row 1: Individual waveforms
        for i, (audio, title, color) in enumerate([
            (audio_loop, 'LOOP Waveform', 'blue'),
            (audio_diff_td, 'DIFFABLE_TD Waveform', 'orange'),
        ]):
            ax = fig.add_subplot(gs[1, i])
            ax.plot(time_samples, audio, lw=0.5, color=color, alpha=0.7)
            ax.set_title(title, fontweight='bold', color=color, fontsize=11)
            ax.set_xlabel('Time (s)', fontsize=10)
            ax.set_ylabel('Amplitude')
            ax.grid(True, alpha=0.3)
        
        # Row 2: Spectrograms
        vmin, vmax = -80, 0
        for i, (f, t, S_db, title, cmap) in enumerate([
            (f_l, t_l, S_l_db, 'LOOP Spectrogram', 'viridis'),
            (f_d, t_d, S_d_db, 'DIFFABLE_TD Spectrogram', 'viridis'),
        ]):
            ax = fig.add_subplot(gs[2, i])
            im = ax.pcolormesh(t, f, S_db, shading='gouraud', cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_ylim([0, min(4000, config.fs/2)])
            ax.set_title(title, fontweight='bold', fontsize=11)
            ax.set_xlabel('Time (s)', fontsize=10)
            ax.set_ylabel('Frequency (Hz)')
            plt.colorbar(im, ax=ax, label='dB')
        
        # Rows 3-5: Parameter trajectories (2 per row, 3 rows)
        param_info = [
            ('f0', f0_modulated.value, 'F0 (Hz)', True),  # True = show pyin
            ('pluck_position', pluck_position_modulated.value, 'Pluck Position', False),
            ('burst_gain', burst_gain_modulated.value, 'Burst Gain', False),
            ('dynamic_level', dynamic_level_modulated.value, 'Dynamic Level', False),
            ('a1', a1_modulated.value, 'Loop Filter (a1)', False),
            ('decay', decay_modulated.value, 'Decay', False),
        ]
        
        for idx, (name, modulated, label, show_pyin) in enumerate(param_info):
            row = 3 + (idx // 2)  # Rows 3, 4, 5
            col = idx % 2  # Columns 0, 1
            ax = fig.add_subplot(gs[row, col])
            color = 'blue' if modulated else 'gray'
            ax.plot(time_frames, params[name], color=color, lw=2, label='Parameter', zorder=2)
            
            # For f0, overlay the pyin estimate
            if show_pyin:
                voiced_mask = f0_pyin_resampled > 0
                ax.plot(time_frames[voiced_mask], f0_pyin_resampled[voiced_mask], 
                       'r--', lw=1.5, alpha=0.7, label='pyin f0', zorder=1)
                ax.legend(loc='best', fontsize=8)
            
            if modulated:
                ax.fill_between(time_frames, params[name], alpha=0.2, color=color)
            ax.set_title(f"{label} {'(LFO)' if modulated else ''}", 
                        fontsize=11, fontweight='bold', color=color if modulated else 'black')
            ax.grid(True, alpha=0.3)
            ax.set_xlabel('Time (s)', fontsize=10)
        
        # Modulated parameters summary
        modulated_params = [name for name, mod, _, _ in param_info if mod]
        mod_str = ', '.join(modulated_params) if modulated_params else 'None'
        
        fig.suptitle(
            f"LOOP vs DIFFABLE_TIME_DOMAIN Comparison\n"
            f"LFO: {lfo_rate.value:.1f}Hz @ {lfo_amplitude.value:.0%} → [{mod_str}]",
            fontsize=14, fontweight='bold', y=0.995
        )
        
        plt.tight_layout(rect=[0, 0, 1, 0.99])
        plt.show()
        
        # ===== STATISTICS =====
        audio_diff = audio_loop - audio_diff_td
        rms_loop = np.sqrt(np.mean(audio_loop**2))
        rms_diff = np.sqrt(np.mean(audio_diff**2))
        snr = 20 * np.log10(rms_loop / (rms_diff + 1e-10))
        max_abs_diff = np.max(np.abs(audio_diff))
        correlation = np.corrcoef(audio_loop, audio_diff_td)[0, 1]
        
        print("\n" + "="*60)
        print("📊 COMPARISON STATISTICS")
        print("="*60)
        print(f"  Signal-to-Noise Ratio (SNR): {snr:.2f} dB")
        print(f"  RMS Error:                   {rms_diff:.6f}")
        print(f"  Max Absolute Difference:     {max_abs_diff:.6f}")
        print(f"  Correlation:                 {correlation:.8f}")
        print(f"  Implementations Match:       {'✅ YES' if snr > 60 else '⚠️ DIFFERS'}")
        print("="*60)
        
        # ===== AUDIO PLAYBACK =====
        print("\n🔊 Audio Playback:")
        print("\nLOOP Implementation:")
        display(ipd.Audio(audio_loop, rate=config.fs))
        
        print("\nDIFFABLE_TIME_DOMAIN Implementation:")
        display(ipd.Audio(audio_diff_td, rate=config.fs))

generate_button.on_click(generate_comparison)

In [5]:
# Organize controls into a nice layout
lfo_box = widgets.VBox([
    widgets.HTML("<h3>🌊 LFO Settings</h3>"),
    lfo_rate,
    lfo_amplitude,
])

general_box = widgets.VBox([
    widgets.HTML("<h3>⚙️ General</h3>"),
    duration_slider,
    trigger_rate,
])

params_box = widgets.VBox([
    widgets.HTML("<h3>🎛️ Parameters (Center + LFO Routing)</h3>"),
    widgets.HBox([f0_center, f0_modulated]),
    widgets.HBox([pluck_position_center, pluck_position_modulated]),
    widgets.HBox([burst_gain_center, burst_gain_modulated]),
    widgets.HBox([dynamic_level_center, dynamic_level_modulated]),
    widgets.HBox([a1_center, a1_modulated]),
    widgets.HBox([decay_center, decay_modulated]),
])

controls = widgets.VBox([
    widgets.HTML("<h2>🎸 LOOP vs DIFFABLE_TIME_DOMAIN Comparison</h2>"),
    widgets.HBox([lfo_box, general_box]),
    params_box,
    widgets.HTML("<br>"),
    generate_button,
], layout=widgets.Layout(padding='10px', border='1px solid #ccc'))

display(controls, output)
print("\n💡 Tip: Check the 'Route LFO' boxes next to parameters to enable LFO modulation, then click Generate!")

Output()


💡 Tip: Check the 'Route LFO' boxes next to parameters to enable LFO modulation, then click Generate!
